In [ ]:
# =========================
# MODEL — ONE CELL, MEMORY-SAFE, VALIDATION NAMING
# XGBoost + CNN-LSTM + Ridge Ensemble
# =========================

import os
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
INPUT_PATH = "/user/data/feature_engineering/feature_panel_model_ready"   # sửa nếu cần
OUT_DIR = "/workspace/code/model"                                # sửa nếu cần

TIME_COL = "pickup_bin_30m"
LOC_COL = "PULocationID"
TARGET_COL = "pickup_demand"

SEQ_WINDOW = 48           # nếu vẫn thiếu RAM, giảm xuống 24
BATCH_SIZE = 128          # an toàn RAM hơn 256
EPOCHS = 1
RANDOM_STATE = 42
EARLY_STOPPING_PATIENCE = 8

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2) REPRODUCIBILITY + SPARK
# ------------------------------------------------------------
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

spark = SparkSession.builder.appName("FinalModelTrainingMemorySafeValidation").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------------------
# 3) HELPERS
# ------------------------------------------------------------
def unique_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def clip_nonnegative(arr):
    arr = np.asarray(arr, dtype=np.float64)
    arr = np.where(np.isnan(arr), 0.0, arr)
    return np.maximum(arr, 0.0)

def mape_nonzero(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > 0
    if mask.sum() == 0:
        return None
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_nonzero": mape_nonzero(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }

def build_cnn_lstm_model(seq_window, n_features):
    inp = layers.Input(shape=(seq_window, n_features))

    x = layers.Conv1D(filters=64, kernel_size=3, padding="causal", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv1D(filters=64, kernel_size=3, padding="causal", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.10)(x)

    out = layers.Dense(1, activation="linear")(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mae",
        metrics=["mae"]
    )
    return model

def normalize_split_values(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "val": "validation",
        "valid": "validation",
        "validation": "validation",
        "train": "train",
        "test": "test",
    }
    return s.map(lambda x: mapping.get(x, x))

def build_seq_dataset_by_split(
    df_pd,
    seq_features,
    target_col,
    time_col,
    loc_col,
    split_col,
    window
):
    valid_splits = ("train", "validation", "test")
    counts = {s: 0 for s in valid_splits}

    grouped = list(df_pd.groupby(loc_col, sort=False))

    # Pass 1: count
    for _, g in grouped:
        g = g.sort_values(time_col, kind="stable")
        n_rows = len(g)
        if n_rows <= window:
            continue
        split_vals = g[split_col].astype(str).to_numpy()
        for i in range(window, n_rows):
            s = split_vals[i]
            if s in counts:
                counts[s] += 1

    total_samples = sum(counts.values())
    if total_samples == 0:
        raise ValueError("Không tạo được sequence samples nào. Kiểm tra lại dữ liệu hoặc SEQ_WINDOW.")

    n_features = len(seq_features)

    storage = {}
    for s in valid_splits:
        storage[s] = {
            "X": np.zeros((counts[s], window, n_features), dtype=np.float32),
            "y": np.zeros(counts[s], dtype=np.float32),
            "locs": np.zeros(counts[s], dtype=np.int64),
            "times": np.empty(counts[s], dtype="datetime64[ns]"),
            "idx": 0,
        }

    # Pass 2: fill
    for loc_id, g in grouped:
        g = g.sort_values(time_col, kind="stable")
        n_rows = len(g)
        if n_rows <= window:
            continue

        vals = g[seq_features].to_numpy(dtype=np.float32, copy=True)
        targets = g[target_col].to_numpy(dtype=np.float32, copy=True)
        t_vals = g[time_col].to_numpy()
        s_vals = g[split_col].astype(str).to_numpy()

        for i in range(window, n_rows):
            s = s_vals[i]
            if s not in storage:
                continue

            idx = storage[s]["idx"]
            storage[s]["X"][idx] = vals[i - window:i]
            storage[s]["y"][idx] = targets[i]
            storage[s]["locs"][idx] = int(loc_id)
            storage[s]["times"][idx] = t_vals[i]
            storage[s]["idx"] += 1

    def pack(split_name):
        meta = pd.DataFrame({
            loc_col: storage[split_name]["locs"],
            time_col: storage[split_name]["times"],
        })
        meta[split_col] = split_name
        return storage[split_name]["X"], storage[split_name]["y"], meta

    return (*pack("train"), *pack("validation"), *pack("test"), counts)

def scale_seq_inplace(X, mean_vec, std_vec):
    X -= mean_vec.reshape(1, 1, -1)
    X /= std_vec.reshape(1, 1, -1)
    return X

def inverse_y_scale(arr, y_min, y_denom):
    arr = np.asarray(arr, dtype=np.float32)
    return arr * y_denom + y_min

# ------------------------------------------------------------
# 4) READ SPARK DATAFRAME
# ------------------------------------------------------------
df = spark.read.parquet(INPUT_PATH)

if "dataset_split" in df.columns:
    SPLIT_COL = "dataset_split"
elif "split" in df.columns:
    SPLIT_COL = "split"
else:
    raise ValueError("Input parquet phải có cột dataset_split hoặc split.")

required_core_cols = [TIME_COL, LOC_COL, SPLIT_COL, TARGET_COL]
missing_core = [c for c in required_core_cols if c not in df.columns]
if missing_core:
    raise ValueError(f"Thiếu cột lõi: {missing_core}")

candidate_tabular_features = [
    "hour", "minute", "half_hour_slot", "day_of_week", "week_of_year", "month", "year",
    "is_weekday", "is_weekend",
    "demand_t_1", "demand_t_2", "demand_t_3", "demand_t_4", "demand_t_5",
    "ha_output", "ewma_output",
    "rolling_max_24h", "rolling_min_24h", "rolling_mean_24h", "rolling_std_24h", "rolling_skew_24h",
    "cluster_id",
    "cluster_Diff_t_1", "cluster_Mean_diff_24h", "cluster_Std_diff_24h",
    "intra_cluster_similarity", "inter_cluster_similarity"
]

candidate_seq_features = [
    TARGET_COL,
    "ewma_output", "ha_output",
    "rolling_mean_24h", "rolling_std_24h", "rolling_skew_24h",
    "cluster_Diff_t_1", "cluster_Mean_diff_24h", "cluster_Std_diff_24h",
    "is_weekday", "is_weekend", "day_of_week", "half_hour_slot"
]

tabular_features = unique_preserve_order([c for c in candidate_tabular_features if c in df.columns])
seq_features = unique_preserve_order([c for c in candidate_seq_features if c in df.columns])

if len(tabular_features) == 0:
    raise ValueError("Không tìm thấy tabular feature nào hợp lệ.")
if len(seq_features) == 0:
    raise ValueError("Không tìm thấy sequence feature nào hợp lệ.")

print("=" * 110)
print("MODEL TRAINING START")
print("=" * 110)
print(f"INPUT_PATH      : {INPUT_PATH}")
print(f"OUT_DIR         : {OUT_DIR}")
print(f"SPLIT_COL       : {SPLIT_COL}")
print(f"TARGET_COL      : {TARGET_COL}")
print(f"SEQ_WINDOW      : {SEQ_WINDOW}")
print(f"BATCH_SIZE      : {BATCH_SIZE}")
print(f"TAB FEATURES    : {len(tabular_features)}")
print(f"SEQ FEATURES    : {len(seq_features)}")

# ------------------------------------------------------------
# 5) TABULAR BRANCH
# ------------------------------------------------------------
tab_cols = unique_preserve_order([TIME_COL, LOC_COL, SPLIT_COL, TARGET_COL] + tabular_features)

pdf_tab = (
    df.select(*tab_cols)
      .orderBy(LOC_COL, TIME_COL)
      .toPandas()
)

pdf_tab[TIME_COL] = pd.to_datetime(pdf_tab[TIME_COL], errors="coerce")
pdf_tab[LOC_COL] = pd.to_numeric(pdf_tab[LOC_COL], errors="coerce")
pdf_tab[TARGET_COL] = pd.to_numeric(pdf_tab[TARGET_COL], errors="coerce")
pdf_tab[SPLIT_COL] = normalize_split_values(pdf_tab[SPLIT_COL])

for c in tabular_features:
    pdf_tab[c] = pd.to_numeric(pdf_tab[c], errors="coerce")

pdf_tab = pdf_tab.dropna(subset=[TIME_COL, LOC_COL, SPLIT_COL, TARGET_COL]).copy()
pdf_tab = pdf_tab[pdf_tab[SPLIT_COL].isin(["train", "validation", "test"])].copy()
pdf_tab[LOC_COL] = pdf_tab[LOC_COL].astype("int64")
pdf_tab[TARGET_COL] = pdf_tab[TARGET_COL].astype("float32")
pdf_tab[tabular_features] = pdf_tab[tabular_features].fillna(0.0).astype("float32")
pdf_tab = pdf_tab.sort_values([LOC_COL, TIME_COL], kind="stable").reset_index(drop=True)

print(f"Rows tabular-ready: {len(pdf_tab):,}")

mask_train_tab = pdf_tab[SPLIT_COL].eq("train").to_numpy()
mask_validation_tab = pdf_tab[SPLIT_COL].eq("validation").to_numpy()
mask_test_tab = pdf_tab[SPLIT_COL].eq("test").to_numpy()

if mask_train_tab.sum() == 0 or mask_validation_tab.sum() == 0 or mask_test_tab.sum() == 0:
    raise ValueError("Thiếu train/validation/test trong dữ liệu tabular.")

X_all_tab = pdf_tab[tabular_features].to_numpy(dtype=np.float32, copy=True)
y_all_tab = pdf_tab[TARGET_COL].to_numpy(dtype=np.float32, copy=True)

X_train_tab = X_all_tab[mask_train_tab]
y_train_tab = y_all_tab[mask_train_tab]
X_validation_tab = X_all_tab[mask_validation_tab]
y_validation_tab = y_all_tab[mask_validation_tab]
X_test_tab = X_all_tab[mask_test_tab]
y_test_tab = y_all_tab[mask_test_tab]

xgb_model = XGBRegressor(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    tree_method="hist",
    eval_metric="mae"
)

xgb_model.fit(
    X_train_tab,
    y_train_tab,
    eval_set=[(X_validation_tab, y_validation_tab)],
    verbose=False
)

train_tab_pred = pdf_tab.loc[mask_train_tab, [LOC_COL, TIME_COL, SPLIT_COL, TARGET_COL]].copy()
train_tab_pred["xgb_pred"] = clip_nonnegative(xgb_model.predict(X_train_tab))

validation_tab_pred = pdf_tab.loc[mask_validation_tab, [LOC_COL, TIME_COL, SPLIT_COL, TARGET_COL]].copy()
validation_tab_pred["xgb_pred"] = clip_nonnegative(xgb_model.predict(X_validation_tab))

test_tab_pred = pdf_tab.loc[mask_test_tab, [LOC_COL, TIME_COL, SPLIT_COL, TARGET_COL]].copy()
test_tab_pred["xgb_pred"] = clip_nonnegative(xgb_model.predict(X_test_tab))

del X_all_tab, y_all_tab, X_train_tab, y_train_tab, X_validation_tab, y_validation_tab, X_test_tab, y_test_tab, pdf_tab
gc.collect()

# ------------------------------------------------------------
# 6) SEQUENCE BRANCH
# ------------------------------------------------------------
seq_cols = unique_preserve_order([TIME_COL, LOC_COL, SPLIT_COL, TARGET_COL] + seq_features)

pdf_seq = (
    df.select(*seq_cols)
      .orderBy(LOC_COL, TIME_COL)
      .toPandas()
)

del df
gc.collect()

pdf_seq[TIME_COL] = pd.to_datetime(pdf_seq[TIME_COL], errors="coerce")
pdf_seq[LOC_COL] = pd.to_numeric(pdf_seq[LOC_COL], errors="coerce")
pdf_seq[TARGET_COL] = pd.to_numeric(pdf_seq[TARGET_COL], errors="coerce")
pdf_seq[SPLIT_COL] = normalize_split_values(pdf_seq[SPLIT_COL])

for c in seq_features:
    pdf_seq[c] = pd.to_numeric(pdf_seq[c], errors="coerce")

pdf_seq = pdf_seq.dropna(subset=[TIME_COL, LOC_COL, SPLIT_COL, TARGET_COL]).copy()
pdf_seq = pdf_seq[pdf_seq[SPLIT_COL].isin(["train", "validation", "test"])].copy()
pdf_seq[LOC_COL] = pdf_seq[LOC_COL].astype("int64")
pdf_seq[TARGET_COL] = pdf_seq[TARGET_COL].astype("float32")
pdf_seq[seq_features] = pdf_seq[seq_features].fillna(0.0).astype("float32")
pdf_seq = pdf_seq.sort_values([LOC_COL, TIME_COL], kind="stable").reset_index(drop=True)

print(f"Rows sequence-ready: {len(pdf_seq):,}")

(
    X_train_seq, y_train_seq_raw, meta_train_seq,
    X_validation_seq, y_validation_seq_raw, meta_validation_seq,
    X_test_seq, y_test_seq_raw, meta_test_seq,
    seq_counts
) = build_seq_dataset_by_split(
    df_pd=pdf_seq,
    seq_features=seq_features,
    target_col=TARGET_COL,
    time_col=TIME_COL,
    loc_col=LOC_COL,
    split_col=SPLIT_COL,
    window=SEQ_WINDOW
)

del pdf_seq
gc.collect()

if len(X_train_seq) == 0 or len(X_validation_seq) == 0 or len(X_test_seq) == 0:
    raise ValueError("Sequence dataset rỗng ở ít nhất một split. Kiểm tra lại SEQ_WINDOW hoặc upstream feature parquet.")

print(f"Sequence train / validation / test: {len(X_train_seq):,} / {len(X_validation_seq):,} / {len(X_test_seq):,}")

# ------------------------------------------------------------
# 7) SCALE SEQUENCE X
# ------------------------------------------------------------
n_seq_features = X_train_seq.shape[-1]

train_2d = X_train_seq.reshape(-1, n_seq_features)
seq_mean = train_2d.mean(axis=0, dtype=np.float64).astype(np.float32)
seq_std = train_2d.std(axis=0, dtype=np.float64).astype(np.float32)
seq_std = np.where(seq_std < 1e-6, 1.0, seq_std).astype(np.float32)

X_train_seq = scale_seq_inplace(X_train_seq, seq_mean, seq_std)
X_validation_seq = scale_seq_inplace(X_validation_seq, seq_mean, seq_std)
X_test_seq = scale_seq_inplace(X_test_seq, seq_mean, seq_std)

# ------------------------------------------------------------
# 8) SCALE TARGET Y
# ------------------------------------------------------------
y_min = float(np.min(y_train_seq_raw))
y_max = float(np.max(y_train_seq_raw))
y_denom = max(y_max - y_min, 1e-6)

y_train_seq = ((y_train_seq_raw - y_min) / y_denom).astype(np.float32)
y_validation_seq = ((y_validation_seq_raw - y_min) / y_denom).astype(np.float32)
y_test_seq = ((y_test_seq_raw - y_min) / y_denom).astype(np.float32)

# ------------------------------------------------------------
# 9) CNN-LSTM TRAIN
# ------------------------------------------------------------
cnn_lstm = build_cnn_lstm_model(SEQ_WINDOW, n_seq_features)

cb = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, EARLY_STOPPING_PATIENCE // 2),
        min_lr=1e-5
    )
]

history = cnn_lstm.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_validation_seq, y_validation_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb,
    verbose=1
)

del y_train_seq, y_validation_seq, y_test_seq
gc.collect()

# ------------------------------------------------------------
# 10) CNN-LSTM PREDICTION (MEMORY-SAFE)
# ------------------------------------------------------------
validation_preds_scaled = cnn_lstm.predict(X_validation_seq, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
test_preds_scaled = cnn_lstm.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0).reshape(-1)

del X_validation_seq, X_test_seq
gc.collect()

train_preds_scaled = cnn_lstm.predict(X_train_seq, batch_size=BATCH_SIZE, verbose=0).reshape(-1)

del X_train_seq
gc.collect()

train_seq_pred = meta_train_seq.copy()
train_seq_pred[TARGET_COL] = y_train_seq_raw
train_seq_pred["cnn_lstm_pred"] = clip_nonnegative(inverse_y_scale(train_preds_scaled, y_min, y_denom))

validation_seq_pred = meta_validation_seq.copy()
validation_seq_pred[TARGET_COL] = y_validation_seq_raw
validation_seq_pred["cnn_lstm_pred"] = clip_nonnegative(inverse_y_scale(validation_preds_scaled, y_min, y_denom))

test_seq_pred = meta_test_seq.copy()
test_seq_pred[TARGET_COL] = y_test_seq_raw
test_seq_pred["cnn_lstm_pred"] = clip_nonnegative(inverse_y_scale(test_preds_scaled, y_min, y_denom))

del train_preds_scaled, validation_preds_scaled, test_preds_scaled
del y_train_seq_raw, y_validation_seq_raw, y_test_seq_raw
del meta_train_seq, meta_validation_seq, meta_test_seq
gc.collect()

# ------------------------------------------------------------
# 11) ENSEMBLE — VALIDATION/TEST FIRST
# ------------------------------------------------------------
key_cols = [LOC_COL, TIME_COL, SPLIT_COL]
meta_features = ["xgb_pred", "cnn_lstm_pred"]

validation_ens = validation_seq_pred.merge(
    validation_tab_pred[key_cols + ["xgb_pred"]],
    on=key_cols,
    how="inner",
    copy=False
)

test_ens = test_seq_pred.merge(
    test_tab_pred[key_cols + ["xgb_pred"]],
    on=key_cols,
    how="inner",
    copy=False
)

if len(validation_ens) == 0 or len(test_ens) == 0:
    raise ValueError("Không merge được dữ liệu ensemble cho validation/test.")

meta_model = Ridge(alpha=1.0)
meta_model.fit(
    validation_ens[meta_features].to_numpy(dtype=np.float32),
    validation_ens[TARGET_COL].to_numpy(dtype=np.float32)
)

validation_ens["ensemble_pred"] = clip_nonnegative(
    meta_model.predict(validation_ens[meta_features].to_numpy(dtype=np.float32))
)
test_ens["ensemble_pred"] = clip_nonnegative(
    meta_model.predict(test_ens[meta_features].to_numpy(dtype=np.float32))
)

# ------------------------------------------------------------
# 12) TRAIN ENSEMBLE LAST
# ------------------------------------------------------------
train_ens = train_seq_pred.merge(
    train_tab_pred[key_cols + ["xgb_pred"]],
    on=key_cols,
    how="inner",
    copy=False
)

if len(train_ens) == 0:
    raise ValueError("Không merge được dữ liệu ensemble cho train.")

train_ens["ensemble_pred"] = clip_nonnegative(
    meta_model.predict(train_ens[meta_features].to_numpy(dtype=np.float32))
)

# ------------------------------------------------------------
# 13) METRICS
# ------------------------------------------------------------
metrics_payload = {
    "xgb_train": regression_metrics(train_tab_pred[TARGET_COL], train_tab_pred["xgb_pred"]),
    "xgb_validation": regression_metrics(validation_tab_pred[TARGET_COL], validation_tab_pred["xgb_pred"]),
    "xgb_test": regression_metrics(test_tab_pred[TARGET_COL], test_tab_pred["xgb_pred"]),

    "cnn_lstm_train": regression_metrics(train_seq_pred[TARGET_COL], train_seq_pred["cnn_lstm_pred"]),
    "cnn_lstm_validation": regression_metrics(validation_seq_pred[TARGET_COL], validation_seq_pred["cnn_lstm_pred"]),
    "cnn_lstm_test": regression_metrics(test_seq_pred[TARGET_COL], test_seq_pred["cnn_lstm_pred"]),

    "ensemble_train": regression_metrics(train_ens[TARGET_COL], train_ens["ensemble_pred"]),
    "ensemble_validation": regression_metrics(validation_ens[TARGET_COL], validation_ens["ensemble_pred"]),
    "ensemble_test": regression_metrics(test_ens[TARGET_COL], test_ens["ensemble_pred"]),
}

# ------------------------------------------------------------
# 14) EXPORT
# ------------------------------------------------------------
xgb_model_path = os.path.join(OUT_DIR, "xgb_model.json")
xgb_model.save_model(xgb_model_path)

cnn_lstm_model_path = os.path.join(OUT_DIR, "cnn_lstm.keras")
cnn_lstm.save(cnn_lstm_model_path)

meta_model_path = os.path.join(OUT_DIR, "ridge_meta_model.joblib")
joblib.dump(meta_model, meta_model_path)

seq_x_stats_path = os.path.join(OUT_DIR, "seq_x_scaler_stats.npz")
np.savez(seq_x_stats_path, mean=seq_mean, std=seq_std)

seq_y_stats_path = os.path.join(OUT_DIR, "seq_y_scaler_stats.npz")
np.savez(seq_y_stats_path, y_min=np.array([y_min], dtype=np.float32), y_max=np.array([y_max], dtype=np.float32))

history_df = pd.DataFrame(history.history)
history_csv_path = os.path.join(OUT_DIR, "cnn_lstm_history.csv")
history_df.to_csv(history_csv_path, index=False)

train_tab_pred.to_csv(os.path.join(OUT_DIR, "train_xgb_predictions.csv"), index=False)
validation_tab_pred.to_csv(os.path.join(OUT_DIR, "validation_xgb_predictions.csv"), index=False)
test_tab_pred.to_csv(os.path.join(OUT_DIR, "test_xgb_predictions.csv"), index=False)

train_seq_pred.to_csv(os.path.join(OUT_DIR, "train_cnn_lstm_predictions.csv"), index=False)
validation_seq_pred.to_csv(os.path.join(OUT_DIR, "validation_cnn_lstm_predictions.csv"), index=False)
test_seq_pred.to_csv(os.path.join(OUT_DIR, "test_cnn_lstm_predictions.csv"), index=False)

train_ens.to_csv(os.path.join(OUT_DIR, "train_ensemble_predictions.csv"), index=False)
validation_ens.to_csv(os.path.join(OUT_DIR, "validation_ensemble_predictions.csv"), index=False)
test_ens.to_csv(os.path.join(OUT_DIR, "test_ensemble_predictions.csv"), index=False)

metrics_path = os.path.join(OUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

run_summary = {
    "input_path": INPUT_PATH,
    "split_col": SPLIT_COL,
    "target_col": TARGET_COL,
    "seq_window": SEQ_WINDOW,
    "batch_size": BATCH_SIZE,
    "tabular_features": tabular_features,
    "seq_features": seq_features,
    "seq_counts": seq_counts,
    "notes": [
        "All val naming replaced by validation.",
        "Legacy split values val/valid are normalized to validation.",
        "Duplicate-column bug fixed by unique_preserve_order().",
        "Sequence dataset built by pre-allocation and split-aware filling.",
        "Validation/Test predicted before Train to reduce post-train memory spike.",
        "Keras predict uses batch_size to avoid one-shot RAM burst.",
        "Validation/Test ensemble built before Train ensemble.",
        "If RAM is still tight, reduce SEQ_WINDOW from 48 to 24 and BATCH_SIZE from 128 to 64."
    ]
}

run_summary_path = os.path.join(OUT_DIR, "run_summary.json")
with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 15) PRINT SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 110)
print("SEQUENCE COUNTS")
print("=" * 110)
for k, v in seq_counts.items():
    print(f"{k}: {v:,}")

print("\n" + "=" * 110)
print("METRICS")
print("=" * 110)
for block, vals in metrics_payload.items():
    print(f"{block}:")
    for k, v in vals.items():
        print(f"  {k}: {v}")

print("\n" + "=" * 110)
print("EXPORTED FILES")
print("=" * 110)
print("xgb_model                      :", xgb_model_path)
print("cnn_lstm_model                 :", cnn_lstm_model_path)
print("meta_model                     :", meta_model_path)
print("seq_x_stats                    :", seq_x_stats_path)
print("seq_y_stats                    :", seq_y_stats_path)
print("history_csv                    :", history_csv_path)
print("metrics                        :", metrics_path)
print("run_summary                    :", run_summary_path)
print("validation_xgb_predictions     :", os.path.join(OUT_DIR, "validation_xgb_predictions.csv"))
print("validation_cnn_lstm_predictions:", os.path.join(OUT_DIR, "validation_cnn_lstm_predictions.csv"))
print("validation_ensemble_predictions:", os.path.join(OUT_DIR, "validation_ensemble_predictions.csv"))


I0000 00:00:1774951002.059595    6452 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774951002.680638    6452 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774951005.089039    6452 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ca966eab-a23f-4636-bd43-184c0e7354d4;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 136ms :: artifacts dl 6ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

MODEL TRAINING START
INPUT_PATH      : /user/data/feature_engineering/feature_panel_model_ready
OUT_DIR         : /user/data/model/paper_aligned
SPLIT_COL       : dataset_split
TARGET_COL      : pickup_demand
SEQ_WINDOW      : 48
BATCH_SIZE      : 128
TAB FEATURES    : 27
SEQ FEATURES    : 13


26/03/31 09:56:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


...

E0000 00:00:1774951084.546302    6452 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1774951085.592867    6452 cpu_allocator_impl.cc:82] Allocation of 773600256 exceeds 10% of free system memory.


2422/2422 ━━━━━━━━━━━━━━━━━━━━ 115s 46ms/step - loss: 0.0227 - mae: 0.0227 - val_loss: 0.0152 - val_mae: 0.0152 - learning_rate: 0.0010


W0000 00:00:1774951221.107346    6452 cpu_allocator_impl.cc:82] Allocation of 773600256 exceeds 10% of free system memory.



SEQUENCE COUNTS
train: 309,936
validation: 95,634
test: 47,322

METRICS
xgb_train:
  MAE: 5.1787831094979495
  RMSE: 8.432082915133124
  MAPE_nonzero: 34.971445665506444
  sMAPE: 40.58860638135038
xgb_validation:
  MAE: 6.219123342380717
  RMSE: 10.950177674942456
  MAPE_nonzero: 33.15920598347867
  sMAPE: 35.90939480151151
xgb_test:
  MAE: 6.607134693074018
  RMSE: 11.78541466265007
  MAPE_nonzero: 36.84927784957907
  sMAPE: 38.07331463837131
cnn_lstm_train:
  MAE: 7.2410583382930795
  RMSE: 13.156731353444947
  MAPE_nonzero: 37.61301879585338
  sMAPE: 45.07081546128665
cnn_lstm_validation:
  MAE: 7.536786889603739
  RMSE: 13.181259469506818
  MAPE_nonzero: 35.82597345130145
  sMAPE: 39.54230448024017
cnn_lstm_test:
  MAE: 7.315748139430069
  RMSE: 12.985087593221508
  MAPE_nonzero: 37.17132532377595
  sMAPE: 40.34436923986335
ensemble_train:
  MAE: 5.390162490841087
  RMSE: 8.782959076003703
  MAPE_nonzero: 37.27395815499076
  sMAPE: 40.19241558052007
ensemble_validation:
  MAE: 6.2

26/03/31 10:03:04 WARN DataStreamer: Exception for BP-1382946106-172.20.0.2-1773904823795:blk_1073742198_1378
java.io.EOFException: Unexpected EOF while trying to read response from server
	at org.apache.hadoop.hdfs.protocolPB.PBHelperClient.vintPrefixed(PBHelperClient.java:521)
	at org.apache.hadoop.hdfs.protocol.datatransfer.PipelineAck.readFields(PipelineAck.java:213)
	at org.apache.hadoop.hdfs.DataStreamer$ResponseProcessor.run(DataStreamer.java:1137)
26/03/31 10:03:05 ERROR TaskSchedulerImpl: Lost executor 1 on 172.20.0.4: worker lost: 172.20.0.4:7078 got disassociated
26/03/31 10:03:05 ERROR TaskSchedulerImpl: Lost executor 0 on 172.20.0.3: worker lost: 172.20.0.3:7078 got disassociated
26/03/31 10:03:05 ERROR AsyncEventQueue: Listener EventLoggingListener threw an exception
java.io.IOException: All datanodes [DatanodeInfoWithStorage[172.20.0.4:9866,DS-c49df8af-fff8-4760-8342-779a315ede28,DISK]] are bad. Aborting...
	at org.apache.hadoop.hdfs.DataStreamer.handleBadDatanode(DataSt